## Testing test.py

In [1]:

import json

import requests
from sample import sample_row
 


In [2]:
root = '../../data' # Raw Data root director
test_csv = f'{root}/test/df_test.csv'
sample_url = sample_row(test_csv)


2025-01-08 21:43:42,017 Generated Random Row: row_55.csv


In [3]:
def url_to_json(url):
    escaped_string = url.replace("'", '"') 
    return json.dumps({"url": escaped_string})

In [4]:
sample_url = sample_row(test_csv)
sample_data = url_to_json(sample_url)  

2025-01-08 21:43:42,034 Generated Random Row: row_66.csv


In [5]:
import pprint
pprint.pp(sample_data)

'{"url": "../../data/test/random_rows/row_66.csv"}'


In [6]:
#url = 'http://localhost:9696/predict'
#response = requests.post(url, json=sample_data)

In [7]:
#response

## Testing predict.py

In [8]:
import pickle
import sys

from tensorflow.keras.models import load_model

# sys.path lists directories that Python searches for modules to import
sys.path.append('../data') 
from data_preparation import dataset

2025-01-08 21:43:42.241648: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-08 21:43:42.245386: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-08 21:43:42.255123: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1736372622.270738   25053 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1736372622.274979   25053 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-08 21:43:42.291922: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

In [9]:
def load_data(test_data):
    root = '../../data' # Raw Data root director
    train_csv = f'{root}/train/df_full_train.csv'
    target = 'price'
    _, _, X_test, y_test = dataset(train_csv, test_data, target, scaler=True)
    return X_test, y_test


def load_xgb(model_file):
    '''Load the pre-trained model and other necessary components'''
    with open(model_file, 'rb') as f:
        model = pickle.load(f)
    print('Successfully loaded XGB Model.')
    return model


def load_ann(model_file, weights=None):
    model = load_model(model_file)
#   model.load_weights(weights)
    print('Successfully loaded ANN Model.')
    return model


def evaluate(model, data):
    '''Process the data, make predictions using the model, and return the results'''
    prediction = model.predict(data)
    print('Evaluating...')
    return float(prediction)


In [10]:
root = '../../models' # Models root directory
model_file_xgb = f'{root}/xgb_v1.pkl'
model_xgb = load_xgb(model_file_xgb)  


Successfully loaded XGB Model.


In [11]:
model_file_ann = f'{root}/ann_v1.h5'
# weights = f'{root}/checkpoints/ann_v1_.ckpt'
model_ann = load_ann(model_file_ann)

2025-01-08 21:43:44,804 Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
Successfully loaded ANN Model.


2025-01-08 21:43:44.724124: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [12]:
models = [
    ('XGB', model_xgb),
    ('ANN', model_ann)
]

# data = request.get_json()
# sample_data = '../../data/test/random_rows/row_38.csv'
data = json.loads(sample_data)
X, y = load_data(test_data=data['url'])
results = {'ACTUAL': float(y)}
for name, model in models:
    results[name] = evaluate(model, X)

Evaluating...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Evaluating...


/tmp/ipykernel_25053/221799082.py:10: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  results = {'ACTUAL': float(y)}
/tmp/ipykernel_25053/1896532898.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(prediction)


In [13]:
results

{'ACTUAL': 28085.0, 'XGB': 30.170032501220703, 'ANN': 27657.00390625}